# AlphaGen US RL Training on Colab

这个 notebook 会在 Colab 中完成以下流程：

1. 挂载 Google Drive。
2. 拉取 AlphaGen 仓库并安装依赖。
3. 用 Qlib 下载或复用缓存的美股日频数据。
4. 按 `train / valid / test` 三段时间切分数据。
5. 运行 AlphaGen 的强化学习训练。
6. 把训练产物和分段评估结果保存回 Google Drive。

默认使用 `SP500` 股票池；如果当前 Qlib 数据目录里没有这个股票池，notebook 会自动从已有股票池里挑选一个可用候选。

In [1]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [2]:
from pathlib import Path

REPO_URL = 'https://github.com/ZZZZkp/alphagen.git'
REPO_BRANCH = 'codex/refresh-runtime-and-smoke-tests'

WORKDIR = Path('/content')
REPO_DIR = WORKDIR / 'alphagen'

DRIVE_ROOT = Path('/content/drive/MyDrive/alphagen_us_rl')
DRIVE_DATA_CACHE = DRIVE_ROOT / 'qlib_data' / 'us_data'
DRIVE_RUNS_DIR = DRIVE_ROOT / 'runs'
LOCAL_QLIB_DIR = WORKDIR / 'qlib_data' / 'us_data'

SEGMENTS = {
    'train': ('2010-01-01', '2018-12-31'),
    'valid': ('2019-01-01', '2020-12-31'),
    'test': ('2021-01-01', '2023-12-31'),
}

SEED = 0
POOL_CAPACITY = 10
TRAINING_STEPS = 20_000
PPO_N_STEPS = 512
BATCH_SIZE = 128
PRINT_EXPR = False

PREFERRED_INSTRUMENTS = ('sp500', 'SP500', 'all', 'ALL')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('Repo:', REPO_URL)
print('Drive root:', DRIVE_ROOT)
print('Segments:', SEGMENTS)

Repo: https://github.com/ZZZZkp/alphagen.git
Drive root: /content/drive/MyDrive/alphagen_us_rl
Segments: {'train': ('2010-01-01', '2018-12-31'), 'valid': ('2019-01-01', '2020-12-31'), 'test': ('2021-01-01', '2023-12-31')}


In [3]:
import os
import subprocess
import sys
from pathlib import Path


def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


if REPO_DIR.exists():
    run(['git', 'fetch', '--all', '--tags'], cwd=str(REPO_DIR))
    run(['git', 'checkout', REPO_BRANCH], cwd=str(REPO_DIR))
    run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=str(REPO_DIR))
else:
    run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])

deps_marker = WORKDIR / '.alphagen_colab_deps_v5'
if not deps_marker.exists():
    run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
    run([
        sys.executable,
        '-m',
        'pip',
        'install',
        '--upgrade',
        '-r',
        str(REPO_DIR / 'requirements.txt'),
    ])
    deps_marker.write_text('ready', encoding='utf-8')
    print('Dependencies installed.')
else:
    print('Dependency bootstrap already completed in this runtime.')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import numpy as np
import pandas as pd
import google.protobuf
import sklearn
import torch
from sb3_contrib.ppo_mask import MaskablePPO

print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('protobuf:', google.protobuf.__version__)
print('scikit-learn:', sklearn.__version__)
print('Torch:', torch.__version__)
print('MaskablePPO import OK:', MaskablePPO.__name__)
print('CUDA available:', torch.cuda.is_available())

+ git clone --depth 1 --branch codex/refresh-runtime-and-smoke-tests https://github.com/ZZZZkp/alphagen.git /content/alphagen
+ /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
+ /usr/bin/python3 -m pip install -r /content/alphagen/requirements.txt
Torch: 2.10.0+cu128
CUDA available: True


In [4]:
import shutil


def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)


def copy_tree(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    reset_dir(dst)
    shutil.copytree(src, dst)


def download_qlib_us_data(target_dir: Path):
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    try:
        run([
            sys.executable,
            '-m',
            'qlib.cli.data',
            'qlib_data',
            '--target_dir',
            str(target_dir),
            '--region',
            'us',
        ])
    except subprocess.CalledProcessError:
        from qlib.tests.data import GetData

        print('CLI 下载失败，尝试调用 qlib.tests.data.GetData().qlib_data(...)')
        GetData().qlib_data(target_dir=str(target_dir), region='us', exists_skip=True)


expected_calendar = DRIVE_DATA_CACHE / 'calendars' / 'day.txt'
if expected_calendar.exists():
    print('Using cached US Qlib data from Google Drive')
    copy_tree(DRIVE_DATA_CACHE, LOCAL_QLIB_DIR)
else:
    print('Downloading US Qlib data to local Colab storage')
    reset_dir(LOCAL_QLIB_DIR)
    download_qlib_us_data(LOCAL_QLIB_DIR)
    produced_calendar = LOCAL_QLIB_DIR / 'calendars' / 'day.txt'
    if not produced_calendar.exists():
        raise FileNotFoundError(
            'Qlib US data download did not create calendars/day.txt. '
            '请检查当前 pyqlib 版本是否还能访问公开数据源，或改成你自己的 provider_uri。'
        )
    copy_tree(LOCAL_QLIB_DIR, DRIVE_DATA_CACHE)

instrument_files = sorted((LOCAL_QLIB_DIR / 'instruments').glob('*.txt'))
available_instruments = [path.stem for path in instrument_files]
print('Available instrument universes:', available_instruments[:20])

SELECTED_INSTRUMENT = next((name for name in PREFERRED_INSTRUMENTS if name in available_instruments), None)
if SELECTED_INSTRUMENT is None:
    raise ValueError(
        f'Could not find a supported US instrument universe in {available_instruments}. '
        '请把 PREFERRED_INSTRUMENTS 改成你的数据目录中真实存在的股票池名称。'
    )

print('Selected instrument universe:', SELECTED_INSTRUMENT)
print('Local Qlib dir:', LOCAL_QLIB_DIR)
print('Drive Qlib cache:', DRIVE_DATA_CACHE)

+ /usr/bin/python3 -m qlib.cli.data qlib_data --target_dir /content/qlib_data/us_data --region us
Available instrument universes: ['all', 'nasdaq100', 'sp500']
Selected instrument universe: sp500
Local Qlib dir: /content/qlib_data/us_data
Drive Qlib cache: /content/drive/MyDrive/alphagen_us_rl/qlib_data/us_data


In [5]:
import torch

from scripts.rl import latest_run, run_single_experiment, status

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
SEGMENT_ORDER = ('train', 'valid', 'test')
SEGMENT_TUPLES = tuple(SEGMENTS[name] for name in SEGMENT_ORDER)

print('Training device:', DEVICE)
print('Training segments:', dict(zip(SEGMENT_ORDER, SEGMENT_TUPLES)))

run_single_experiment(
    seed=SEED,
    instruments=SELECTED_INSTRUMENT,
    pool_capacity=POOL_CAPACITY,
    steps=TRAINING_STEPS,
    qlib_data_path=str(LOCAL_QLIB_DIR),
    qlib_region='us',
    device=DEVICE,
    segments=SEGMENT_TUPLES,
    ppo_n_steps=PPO_N_STEPS,
    batch_size=BATCH_SIZE,
    print_expr=PRINT_EXPR,
)

RUN_DIR = Path(latest_run())
print('Latest run dir:', RUN_DIR)
status(str(RUN_DIR))

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import json
import pandas as pd
import shutil

from alphagen.data.expression import Feature, Ref
from alphagen.models.linear_alpha_pool import MseAlphaPool
from alphagen_qlib.calculator import QLibStockDataCalculator
from alphagen_qlib.stock_data import FeatureType, StockData, initialize_qlib
from alphagen_qlib.utils import load_alpha_pool_by_path

checkpoint_paths = sorted(
    RUN_DIR.glob('*_steps_pool.json'),
    key=lambda path: int(path.name.split('_', 1)[0]),
)
if not checkpoint_paths:
    raise FileNotFoundError(f'No *_steps_pool.json checkpoint found under {RUN_DIR}')

final_pool_path = checkpoint_paths[-1]
exprs, weights = load_alpha_pool_by_path(str(final_pool_path))
initialize_qlib(str(LOCAL_QLIB_DIR), region='us')

close = Feature(FeatureType.CLOSE)
target = Ref(close, -20) / close - 1

rows = []
for split_name, (start_time, end_time) in SEGMENTS.items():
    data = StockData(
        instrument=SELECTED_INSTRUMENT,
        start_time=start_time,
        end_time=end_time,
        device=DEVICE,
    )
    calculator = QLibStockDataCalculator(data, target)
    pool = MseAlphaPool(
        capacity=max(POOL_CAPACITY, len(exprs)),
        calculator=calculator,
        ic_lower_bound=None,
        l1_alpha=5e-3,
        device=DEVICE,
    )
    pool.force_load_exprs(exprs, weights=weights)
    ic, rank_ic = pool.test_ensemble(calculator)
    rows.append(
        {
            'split': split_name,
            'start_time': start_time,
            'end_time': end_time,
            'n_days': int(data.n_days),
            'n_stocks': int(data.n_stocks),
            'ic': float(ic),
            'rank_ic': float(rank_ic),
            'pool_size': len(exprs),
            'checkpoint': final_pool_path.name,
        }
    )

metrics_df = pd.DataFrame(rows)
metrics_json_path = RUN_DIR / 'segment_metrics.json'
metrics_csv_path = RUN_DIR / 'segment_metrics.csv'
config_json_path = RUN_DIR / 'colab_config.json'

metrics_json_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
metrics_df.to_csv(metrics_csv_path, index=False)
config_json_path.write_text(
    json.dumps(
        {
            'repo_url': REPO_URL,
            'repo_branch': REPO_BRANCH,
            'seed': SEED,
            'pool_capacity': POOL_CAPACITY,
            'training_steps': TRAINING_STEPS,
            'ppo_n_steps': PPO_N_STEPS,
            'batch_size': BATCH_SIZE,
            'instrument': SELECTED_INSTRUMENT,
            'qlib_region': 'us',
            'qlib_data_path': str(LOCAL_QLIB_DIR),
            'segments': SEGMENTS,
            'device': str(DEVICE),
            'final_pool_path': final_pool_path.name,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding='utf-8',
)

drive_run_dir = DRIVE_RUNS_DIR / RUN_DIR.name
if drive_run_dir.exists():
    shutil.rmtree(drive_run_dir)
shutil.copytree(RUN_DIR, drive_run_dir)

print(metrics_df)
print('Saved run directory to:', drive_run_dir)
print('Saved metrics JSON to:', drive_run_dir / 'segment_metrics.json')
print('Saved metrics CSV to:', drive_run_dir / 'segment_metrics.csv')

## 调参建议

- Colab 首次运行建议先保持 `TRAINING_STEPS = 20_000` 做一轮冒烟，确认数据、显卡和保存路径都正常。
- 如果你打算做更完整的训练，可以把 `POOL_CAPACITY` 和 `TRAINING_STEPS` 一起提高；仓库当前默认配置里，`pool_capacity=10` 对应的完整训练规模是 `200_000` steps。
- 如果公开 Qlib 数据下载通道暂时不可用，可以把你自己的 US Qlib 二进制数据目录先放到 `Google Drive/alphagen_us_rl/qlib_data/us_data`，这个 notebook 会优先复用该缓存。